# RSNA Pneumonia Detection - Advanced Coding Assignment
----- 
**Student Name:** ISHAAN KUMAR  
**Registration ID:** S24BCAU0183  
**Batch:** B6  
**Objective:** Implementation of an optimized Faster R-CNN model with data augmentation, transfer learning, and comprehensive evaluation metrics.

## 1. System Configuration & GPU Optimization
In this section, we initialize the environment and verify the GPU hardware availability. We use `CUDNN_BENCHMARK` for optimized convolutions and `torch.cuda.empty_cache()` to ensure maximum VRAM availability.

In [ ]:
import os
import sys
import torch
import pandas as pd
import config
from S24BCAU0183_main import get_device, set_seed, train
import argparse

set_seed(config.RANDOM_SEED)
device = get_device()
print(f"Device: {device}")

## 2. Modular Pipeline Setup
We leverage a modular architecture to maintain clean code and professional standards. Each component (data loading, model architecture, training, and evaluation) is handled by its respective module.

In [ ]:
from data_preparation import prepare_data
from model import get_model
from train import train_model
from evaluate import evaluate_model, print_metrics
from metrics_report import print_full_report

## 3. Training and Evaluation
We execute the training pipeline using the **Improved Model** configuration. This includes:
- **Data Augmentation**: Horizontal flipping, color jitter, and random affine transformations.
- **Transfer Learning**: ResNet-50 FPN backbone pretrained on COCO.
- **Optimization**: AdamW optimizer with Automatic Mixed Precision (AMP) for faster training.

In [ ]:
# Define arguments for the training mode
class Args:
    def __init__(self):
        self.mode = 'train'
        self.epochs = 5
        self.batch_size = config.BATCH_SIZE
        self.lr = config.LEARNING_RATE
        self.weight_decay = config.WEIGHT_DECAY
        self.image_size = config.IMAGE_SIZE
        self.num_workers = config.NUM_WORKERS
        self.model_type = 'fasterrcnn'
        self.pretrained = True
        self.augmentation = True
        self.freeze_backbone = True
        self.freeze_layers = 2
        self.optimizer = 'adamw'
        self.scheduler = 'plateau'
        self.early_stopping = True
        self.early_stopping_patience = 5
        self.checkpoint = config.MODEL_SAVE_PATH
        self.num_viz = 8
        self.grad_accum_steps = 1
        self.sample_size = 1000

args = Args()

# Run the training function directly from the modular script
train(args)

## 4. Final Metrics & Visualizations
After training, we generate a comprehensive metrics report and visualize the model's predictions using a score threshold of 0.5.

In [ ]:
import json
from metrics_report import print_full_report

print("\nTraining Complete. Generating final metrics report...")

with open('output/metrics.json', 'r') as f:
    metrics = json.load(f)

print_full_report(metrics)

In [ ]:
import matplotlib.pyplot as plt
import cv2

print("\nGenerating Visualizations...")
viz_dir = 'output/visualizations'
if os.path.exists(viz_dir):
    files = [f for f in os.listdir(viz_dir) if f.endswith('.png')]
    for i, file in enumerate(files[:3]):
        img = cv2.imread(os.path.join(viz_dir, file))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(f"Prediction Sample {i+1}")
        plt.axis('off')
        plt.show()